# 🚕 Project 1: Data Analysis Using Pandas (NYC Taxi Revenue Analysis)

**The Real-World Scenario:**
You are hired as a Data Analysts for the NYC Transport Authority. You are handed a dataset of taxi trips containing pickup times, distances, fares, and payment methods. The data has missing values (like unrecorded drop-off zones) and raw date strings. Your job is to clean the data, calculate new metrics (like trip duration and fare per mile), and build a summary report showing where the most revenue is generated.

### 📝 Learning Objectives

* **Data Ingestion from the Web:** Using `pd.read_csv()` with a URL.
* **Handling Datetimes:** Converting text-based dates into Pandas Datetime objects (a vital real-world skill).
* **Data Cleaning:** Choosing when to `fillna()` vs when to `dropna()`.
* **Feature Engineering:** Doing math between columns (e.g., subtracting times, dividing fare by distance).
* **Business Reporting:** Using `groupby` and `pivot_table` to summarize revenue and tipping behavior.

---

**1: Data Ingestion & Quick Inspection**
*load real data directly from a raw URL.*

In [ ]:
import pandas as pd
import numpy as np

# 1. Load real NYC Taxi Data directly from GitHub (Seaborn's data repository)
url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/taxis.csv'
df_raw = pd.read_csv(url)

print("--- NYC Taxi Dataset Loaded ---")
print(f"Total Trips (Rows): {df_raw.shape[0]}")
print(f"Data Points (Columns): {df_raw.shape[1]}\n")

--- NYC Taxi Dataset Loaded ---
Total Trips (Rows): 6433
Data Points (Columns): 14



In [ ]:
# Display the first 5 rows
display(df_raw.head())

,pickup,dropoff,passengers,distance,fare,tip,tolls,total,color,payment,pickup_zone,dropoff_zone,pickup_borough,dropoff_borough
0,2019-03-23 20:21:09,2019-03-23 20:27:24,1,1.60,7.0,2.15,0.0,12.95,yellow,credit card,Lenox Hill West,UN/Turtle Bay South,Manhattan,Manhattan
1,2019-03-04 16:11:55,2019-03-04 16:19:00,1,0.79,5.0,0.00,0.0,9.30,yellow,cash,Upper West Side South,Upper West Side South,Manhattan,Manhattan
2,2019-03-27 17:53:01,2019-03-27 18:00:25,1,1.37,7.5,2.36,0.0,14.16,yellow,credit card,Alphabet City,West Village,Manhattan,Manhattan
3,2019-03-10 01:23:59,2019-03-10 01:49:51,1,7.70,27.0,6.15,0.0,36.95,yellow,credit card,Hudson Sq,Yorkville West,Manhattan,Manhattan
4,2019-03-30 13:27:42,2019-03-30 13:37:14,3,2.16,9.0,1.10,0.0,13.40,yellow,credit card,Midtown East,Yorkville West,Manhattan,Manhattan


**2: Deep Inspection (Finding the Mess)**
*Before cleaning, we must find where the missing values and incorrect data types are.*

In [ ]:
# 2. Inspecting Data Types and Missing Values
print("--- Data Types & Non-Null Counts ---")
# Notice that 'pickup' and 'dropoff' are 'object' (text), not datetimes!
df_raw.info()

--- Data Types & Non-Null Counts ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6433 entries, 0 to 6432
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   pickup           6433 non-null   object 
 1   dropoff          6433 non-null   object 
 2   passengers       6433 non-null   int64  
 3   distance         6433 non-null   float64
 4   fare             6433 non-null   float64
 5   tip              6433 non-null   float64
 6   tolls            6433 non-null   float64
 7   total            6433 non-null   float64
 8   color            6433 non-null   object 
 9   payment          6389 non-null   object 
 10  pickup_zone      6407 non-null   object 
 11  dropoff_zone     6388 non-null   object 
 12  pickup_borough   6407 non-null   object 
 13  dropoff_borough  6388 non-null   object 
dtypes: float64(5), int64(1), object(8)
memory usage: 703.7+ KB


In [ ]:
print("\n--- Missing Values Count ---")
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])


--- Missing Values Count ---
payment            44
pickup_zone        26
dropoff_zone       45
pickup_borough     26
dropoff_borough    45
dtype: int64


**3: Data Cleaning**
*Real data is messy. We will drop rows with missing zones (since we can't guess where a taxi went) and fill missing payment types with the most common method.*

In [ ]:
# 3. Data Cleaning
df_clean = df_raw.copy()

# A. Drop rows where we don't know the pickup or dropoff zone (Critical missing data)
df_clean.dropna(subset=['pickup_zone', 'dropoff_zone'], inplace=True)

print(f"Rows remaining after dropping missing zones: {df_clean.shape[0]}")

Rows remaining after dropping missing zones: 6383


In [ ]:
# B. Fill missing 'payment' types with the mode (most frequent payment method)
most_common_payment = df_clean['payment'].mode()[0]
print(f"Most common payment method: {most_common_payment}")
df_clean['payment'] = df_clean['payment'].fillna(most_common_payment)

Most common payment method: credit card


In [ ]:
# C. Remove trips with 0 distance or 0 fare (Data entry errors / canceled rides)
df_clean = df_clean[(df_clean['distance'] > 0) & (df_clean['fare'] > 0)]

print(f"Rows remaining after cleaning: {df_clean.shape[0]}")

Rows remaining after cleaning: 6347


**4: Feature Engineering (The Power of Datetimes)**
*Converting strings to dates and calculating new business metrics.*

In [ ]:
# 4. Feature Engineering

# A. Convert 'pickup' and 'dropoff' from text strings into proper Pandas Datetime objects
df_clean['pickup'] = pd.to_datetime(df_clean['pickup'])
df_clean['dropoff'] = pd.to_datetime(df_clean['dropoff'])

In [ ]:
# B. Calculate Trip Duration in minutes
# Subtracting datetimes gives a 'timedelta'. We convert it to total seconds, then divide by 60.
df_clean['duration_minutes'] = (df_clean['dropoff'] - df_clean['pickup']).dt.total_seconds() / 60
df_clean['duration_minutes'] = df_clean['duration_minutes'].round(2)

In [ ]:
# C. Calculate Fare per Mile (Economic metric)
df_clean['fare_per_mile'] = (df_clean['fare'] / df_clean['distance']).round(2)

In [ ]:
# D. Extract the Hour of the day the trip started
# df_clean['pickup_hour'] = df_clean['pickup'].dt.hour
# df_clean['pickup_hour']
display(df_clean[['pickup', 'duration_minutes', 'distance', 'fare', 'fare_per_mile']].head())

,pickup,duration_minutes,distance,fare,fare_per_mile
0,2019-03-23 20:21:09,6.25,1.60,7.0,4.38
1,2019-03-04 16:11:55,7.08,0.79,5.0,6.33
2,2019-03-27 17:53:01,7.40,1.37,7.5,5.47
3,2019-03-10 01:23:59,25.87,7.70,27.0,3.51
4,2019-03-30 13:27:42,9.53,2.16,9.0,4.17


**5: Business Intelligence & Reporting**
*Aggregating the clean, engineered data into readable insights.*

In [ ]:
# 5. Extracting Business Insights

# Insight 1: Which Borough generates the highest average fare and tip?
borough_summary = df_clean.groupby('pickup_borough').agg(
    Total_Trips=('pickup', 'count'),
    Avg_Fare=('fare', 'mean'),
    Avg_Tip=('tip', 'mean')
).round(2).sort_values(by='Total_Trips', ascending=False)

print("--- Taxi Metrics by Pickup Borough ---")
display(borough_summary)

--- Taxi Metrics by Pickup Borough ---


,Total_Trips,Avg_Fare,Avg_Tip
pickup_borough,,,
Manhattan,5243,11.07,1.93
Queens,633,24.49,3.09
Brooklyn,374,16.46,0.99
Bronx,97,20.67,0.15


In [ ]:
# Insight 2: Tipping behavior based on Payment Method and Taxi Color (Pivot Table)
tip_pivot = pd.pivot_table(
    df_clean,
    values='tip',
    index='payment',
    columns='color',
    aggfunc='mean'
).round(2)

print("\n--- Average Tip by Payment Method & Taxi Color ---")
display(tip_pivot)


--- Average Tip by Payment Method & Taxi Color ---


color,green,yellow
payment,,
cash,0.00,0.00
credit card,1.37,2.92
